# DocRestore: NAFNet-TextAware + DocRes

Two models trained with TextAware OCR-guided loss (L1 + 0.1×Perceptual + 1.0×TextAware) at 512×512 for 50 epochs.

| Cell | Description |
|------|-------------|
| 1 | GPU check |
| 2 | Install dependencies |
| 3 | Load repository |
| 4 | Configure training |
| 5 | Generate data + split |
| 6 | Train NAFNet-TextAware |
| 7 | Train DocRes |
| 8 | Evaluate |
| 9 | Results table |
| 10 | Domain gap |
| 11 | Graphs |
| 12 | Zip + download |

## Cell 1: GPU Check

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    print('No GPU found. Set Accelerator to GPU T4 x2 and restart.')
    sys.exit(1)
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')

## Cell 2: Install Dependencies

In [ ]:
!pip install -q augraphy pymupdf pytesseract scikit-image pyyaml pandas scikit-learn
!apt-get install -qq -y tesseract-ocr

import importlib
for pkg in ['augraphy', 'fitz', 'pytesseract', 'skimage', 'yaml', 'sklearn']:
    try:
        importlib.import_module(pkg)
        print(f'  ok  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}')

## Cell 3: Load Repository

In [ ]:
import os, shutil, sys
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
matches = list(KAGGLE_INPUT.rglob('train/train_nafnet.py'))
if not matches:
    print('Repo not found. Attach doc-restore-code dataset.')
    sys.exit(1)

DATASET_SRC = matches[0].parent.parent
REPO_DIR  = Path('/kaggle/working/doc-restore')
CKPT_DIR  = Path('/kaggle/working/checkpoints')

# Skip copy if code already present — preserves data/shabby between sessions
if (REPO_DIR / 'train' / 'train_nafnet.py').exists():
    print('Repo already present, skipping copy.')
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(DATASET_SRC, REPO_DIR)
    print('Repo copied from dataset.')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f'Repo: {REPO_DIR}')
print(f'Checkpoints: {CKPT_DIR}')
print(f'Python files: {len(list(REPO_DIR.rglob("*.py")))}')

## Cell 4: Configure Training

Both models use TextAware OCR-guided loss. `lambda_text=1.0` doubles the weight on text-stroke edges vs the previous run.

In [ ]:
import os, yaml
from pathlib import Path

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

CKPT_DIR   = Path('/kaggle/working/checkpoints')
INPUT_SIZE = 512
BATCH_SIZE = 2
EPOCHS     = 50

base_cfg = {
    'epochs':            EPOCHS,
    'batch_size':        BATCH_SIZE,
    'learning_rate':     0.0002,
    'input_size':        INPUT_SIZE,
    'loss':              'text_aware_combined',
    'lambda_perceptual': 0.1,
    'lambda_text':       1.0,
    'scheduler':         'cosine',
    'train_csv':         'data/train.csv',
    'val_csv':           'data/val.csv',
    'checkpoint_dir':    str(CKPT_DIR),
}

for cfg_path, model_key in [
    ('configs/kaggle_nafnet_textaware.yaml', 'nafnet'),
    ('configs/kaggle_docres.yaml',           'docres'),
]:
    cfg = {**base_cfg, 'model': model_key}
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    print(f'Written: {cfg_path}  (model={model_key})')

print(f'INPUT_SIZE={INPUT_SIZE}  BATCH_SIZE={BATCH_SIZE}  EPOCHS={EPOCHS}  lambda_text=1.0')

## Cell 5: Generate Data + Split

Downloads arXiv PDFs and applies Augraphy degradation pipeline. Skips if already present. Uses 70/15/15 train/val/test split for a larger, more stable validation set.

In [ ]:
import subprocess
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Check working dir first
clean_imgs    = list(Path('data/shabby/clean').glob('*.jpg'))  + list(Path('data/shabby/clean').glob('*.png'))
degraded_imgs = list(Path('data/shabby/degraded').glob('*.jpg')) + list(Path('data/shabby/degraded').glob('*.png'))

if clean_imgs and degraded_imgs:
    print(f'Data present: {len(clean_imgs)} clean | {len(degraded_imgs)} degraded')
else:
    # Kaggle auto-extracts datasets — look for clean/degraded dirs in /kaggle/input
    input_cleans    = [d for d in Path('/kaggle/input').rglob('clean')
                       if d.is_dir() and list(d.glob('*.jpg')) + list(d.glob('*.png'))]
    input_degradeds = [d for d in Path('/kaggle/input').rglob('degraded')
                       if d.is_dir() and list(d.glob('*.jpg')) + list(d.glob('*.png'))]
    if input_cleans and input_degradeds:
        clean_imgs    = list(input_cleans[0].glob('*.jpg'))    + list(input_cleans[0].glob('*.png'))
        degraded_imgs = list(input_degradeds[0].glob('*.jpg')) + list(input_degradeds[0].glob('*.png'))
        print(f'Using input dataset: {len(clean_imgs)} clean | {len(degraded_imgs)} degraded')
    else:
        print('Downloading data...')
        subprocess.run(['python', 'data/download_shabby.py'], check=True)
        clean_imgs    = list(Path('data/shabby/clean').glob('*.jpg'))  + list(Path('data/shabby/clean').glob('*.png'))
        degraded_imgs = list(Path('data/shabby/degraded').glob('*.jpg')) + list(Path('data/shabby/degraded').glob('*.png'))
        print(f'Done: {len(clean_imgs)} clean | {len(degraded_imgs)} degraded')

# Build split if missing
if all(Path(f'data/{s}.csv').exists() for s in ['train', 'val', 'test']):
    print('Split CSVs already present.')
else:
    clean_map = {p.stem: p for p in clean_imgs}
    deg_map   = {p.stem: p for p in degraded_imgs}
    common    = sorted(clean_map.keys() & deg_map.keys())
    rows = [{'clean_path': str(clean_map[s]), 'degraded_path': str(deg_map[s])} for s in common]
    df   = pd.DataFrame(rows)
    train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
    val_df,  test_df  = train_test_split(temp_df, test_size=0.50, random_state=42)
    train_df.to_csv('data/train.csv', index=False)
    val_df.to_csv('data/val.csv',     index=False)
    test_df.to_csv('data/test.csv',   index=False)
    print(f'Split 70/15/15: train={len(train_df)} | val={len(val_df)} | test={len(test_df)}')

# Filter to only existing files
for split in ['train', 'val', 'test']:
    p  = Path(f'data/{split}.csv')
    df = pd.read_csv(p)
    before = len(df)
    mask = (df['clean_path'].apply(lambda x: Path(x).exists()) &
            df['degraded_path'].apply(lambda x: Path(x).exists()))
    df[mask].to_csv(p, index=False)
    print(f'{split}: {before} -> {len(df[mask])} valid pairs')

## Cell 6: Train NAFNet-TextAware

NAFNet with L1 + Perceptual + TextAware loss. 50 epochs at 512×512.

In [ ]:
!python train/train_nafnet.py \
    --config         configs/kaggle_nafnet_textaware.yaml \
    --run-name       nafnet_textaware \
    --checkpoint-dir {CKPT_DIR}
print('\nNAFNet TextAware: DONE')

## Cell 7: Train DocRes

DocRes with the same TextAware loss. Runs for as many epochs as the session allows.

In [ ]:
!python train/train_docres.py \
    --config         configs/kaggle_docres.yaml \
    --run-name       docres \
    --checkpoint-dir {CKPT_DIR}
print('\nDocRes: DONE')

## Cell 8: Evaluate

In [ ]:
!python eval/run_baseline.py \
    --test-csv data/test.csv \
    --out-dir  /kaggle/working/eval/baseline

!python eval/run_eval.py \
    --model      nafnet \
    --checkpoint {CKPT_DIR}/nafnet_textaware_best.pth \
    --test-csv   data/test.csv \
    --out-dir    /kaggle/working/eval/nafnet_textaware

!python eval/run_eval.py \
    --model      docres \
    --checkpoint {CKPT_DIR}/docres_best.pth \
    --test-csv   data/test.csv \
    --out-dir    /kaggle/working/eval/docres

## Cell 9: Results Table

In [ ]:
import pandas as pd
from pathlib import Path

models = [
    ('Baseline (no model)',  '/kaggle/working/eval/baseline/results.csv'),
    ('NAFNet-TextAware',     '/kaggle/working/eval/nafnet_textaware/results.csv'),
    ('DocRes-TextAware',     '/kaggle/working/eval/docres/results.csv'),
]

print(f'{"Model":<28}  {"PSNR":>8}  {"SSIM":>8}  {"CER":>8}')
print('-' * 58)
for name, path in models:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f'{name:<28}  {df["psnr"].mean():>8.2f}  {df["ssim"].mean():>8.4f}  {df["cer"].mean():>8.4f}')
    else:
        print(f'{name:<28}  not found')

## Cell 10: Domain Gap: NoisyOffice

In [ ]:
import re, subprocess
import pandas as pd
from pathlib import Path

noisy_root = Path('data/noisy')
if not (noisy_root / 'degraded').exists():
    subprocess.run(['python', 'data/download_noisy.py', '--dest', 'data/noisy'], check=True)

rows = []
for deg_path in sorted((noisy_root / 'degraded').glob('*.png')):
    clean_stem = re.sub(r'_[Nn]oise[^_]*_', '_Clean_', deg_path.stem)
    clean_path = noisy_root / 'clean' / f'{clean_stem}.png'
    if clean_path.exists():
        rows.append({'degraded_path': str(deg_path), 'clean_path': str(clean_path)})

noisy_csv = Path('data/noisyoffice_test.csv')
pd.DataFrame(rows).to_csv(noisy_csv, index=False)
print(f'NoisyOffice: {len(rows)} pairs')

!python eval/run_eval.py \
    --model      nafnet \
    --checkpoint {CKPT_DIR}/nafnet_textaware_best.pth \
    --test-csv   {noisy_csv} \
    --out-dir    /kaggle/working/eval/noisyoffice

df_n = pd.read_csv('/kaggle/working/eval/noisyoffice/results.csv')
df_s = pd.read_csv('/kaggle/working/eval/nafnet_textaware/results.csv')
print(f'Synthetic:   PSNR={df_s["psnr"].mean():.2f}  SSIM={df_s["ssim"].mean():.4f}  CER={df_s["cer"].mean():.4f}')
print(f'NoisyOffice: PSNR={df_n["psnr"].mean():.2f}  SSIM={df_n["ssim"].mean():.4f}  CER={df_n["cer"].mean():.4f}')
print(f'Gap (PSNR):  {df_s["psnr"].mean() - df_n["psnr"].mean():.2f} dB')

## Cell 11: Graphs

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from IPython.display import Image as IPImage, display

GRAPHS_DIR = Path('/kaggle/working/graphs')
GRAPHS_DIR.mkdir(exist_ok=True)

# Loss curves
model_cfgs = [
    ('nafnet_textaware', 'NAFNet-TextAware', '#4CAF50'),
    ('docres',           'DocRes-TextAware',  '#9C27B0'),
]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (run_name, label, color) in zip(axes, model_cfgs):
    log_path = CKPT_DIR / f'{run_name}_loss_log.csv'
    if not log_path.exists():
        ax.text(0.5, 0.5, 'No log', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label); continue
    df = pd.read_csv(log_path, header=None, names=['epoch','train_loss','val_loss'])
    if str(df.iloc[0]['epoch']).strip() == 'epoch':
        df = df.iloc[1:].reset_index(drop=True)
    df = df.apply(pd.to_numeric)
    ax.plot(df['epoch'], df['train_loss'], color=color, lw=1.8, label='Train')
    ax.plot(df['epoch'], df['val_loss'],   color=color, lw=1.8, ls='--', alpha=0.8, label='Val')
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.suptitle('Training & Validation Loss', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.close(); print('Saved: loss_curves.png')

# Metrics bar
results_map = [
    ('Baseline',         '/kaggle/working/eval/baseline/results.csv',         '#9E9E9E'),
    ('NAFNet-TextAware', '/kaggle/working/eval/nafnet_textaware/results.csv', '#4CAF50'),
    ('DocRes-TextAware', '/kaggle/working/eval/docres/results.csv',           '#9C27B0'),
]
labels, psnrs, ssims, cers, colors = [], [], [], [], []
for label, path, color in results_map:
    if Path(path).exists():
        df = pd.read_csv(path)
        labels.append(label); colors.append(color)
        psnrs.append(df['psnr'].mean()); ssims.append(df['ssim'].mean()); cers.append(df['cer'].mean())

if labels:
    x = range(len(labels))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))
    def _bar(ax, vals, title, ylabel, fmt, pad):
        bars = ax.bar(x, vals, color=colors, edgecolor='k', linewidth=0.6)
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold'); ax.grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+pad, fmt.format(val),
                    ha='center', va='bottom', fontsize=8)
    _bar(ax1, psnrs, 'PSNR (dB)',          'PSNR (dB)', '{:.2f}', 0.3)
    _bar(ax2, ssims, 'SSIM',               'SSIM',       '{:.3f}', 0.005)
    _bar(ax3, cers,  'CER (lower=better)', 'CER',        '{:.3f}', 0.005)
    fig.suptitle('NAFNet-TextAware vs DocRes (50 epochs, 512x512)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / 'metrics_bar.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved: metrics_bar.png')

# Domain gap chart
synth_p = Path('/kaggle/working/eval/nafnet_textaware/results.csv')
noisy_p = Path('/kaggle/working/eval/noisyoffice/results.csv')
if synth_p.exists() and noisy_p.exists():
    df_s = pd.read_csv(synth_p); df_n = pd.read_csv(noisy_p)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    domains = ['Synthetic\n(train domain)', 'NoisyOffice\n(real scans)']
    bar_colors = ['#4CAF50', '#F44336']
    for ax, metric, ylabel in [(ax1,'psnr','PSNR (dB)'), (ax2,'ssim','SSIM')]:
        vals = [df_s[metric].mean(), df_n[metric].mean()]
        bars = ax.bar(domains, vals, color=bar_colors, edgecolor='k', linewidth=0.6, width=0.5)
        ax.set_ylabel(ylabel); ax.grid(axis='y', alpha=0.3)
        ax.set_title(f'{ylabel} (gap={abs(vals[0]-vals[1]):.2f})', fontweight='bold')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    fig.suptitle('Domain Gap: NAFNet-TextAware', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(GRAPHS_DIR / 'domain_gap.png', dpi=150, bbox_inches='tight')
    plt.close(); print('Saved: domain_gap.png')

# Display all
for fname in ['loss_curves.png', 'metrics_bar.png', 'domain_gap.png']:
    p = GRAPHS_DIR / fname
    if p.exists():
        print(f'\n{fname}'); display(IPImage(str(p)))

## Cell 12: Zip Results

Download from Output tab -> `docrestore_results.zip`

If too large (>1GB), split with:
```python
import math
from pathlib import Path
data = Path('/kaggle/working/docrestore_results.zip').read_bytes()
chunk = 500*1024*1024
for i in range(math.ceil(len(data)/chunk)):
    Path(f'/kaggle/working/part{i+1}.zip').write_bytes(data[i*chunk:(i+1)*chunk])
    print(f'part{i+1}.zip')
```
Reassemble locally: `cat part*.zip > docrestore_results.zip`

In [ ]:
import shutil, zipfile
from pathlib import Path

CKPT_DIR   = Path('/kaggle/working/checkpoints')
GRAPHS_DIR = Path('/kaggle/working/graphs')
out        = Path('/kaggle/working/results')
out.mkdir(exist_ok=True)

ckpt_out = out / 'checkpoints'
ckpt_out.mkdir(exist_ok=True)
for f in CKPT_DIR.glob('*.pth'):
    shutil.copy(f, ckpt_out / f.name)
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')
for f in CKPT_DIR.glob('*.csv'):
    shutil.copy(f, ckpt_out / f.name)

shutil.copytree('/kaggle/working/eval', out / 'eval',   dirs_exist_ok=True)
shutil.copytree(str(GRAPHS_DIR),        str(out / 'graphs'), dirs_exist_ok=True)

zip_path = Path('/kaggle/working/docrestore_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in out.rglob('*'):
        if fp.is_file():
            zf.write(fp, fp.relative_to('/kaggle/working'))

print(f'\nZip: {zip_path}  ({zip_path.stat().st_size/1e6:.0f} MB)')
print('Output tab -> docrestore_results.zip -> Download')